In [ ]:
# hafta 9: tokenizer (BPE)

In [ ]:
!pip install -q regex tiktoken

## task 1
Metni byte'lara çevir. Türkçe harfler içeren bir cümle seç (ç, ğ, ı, ö, ş, ü). Her harfin Unicode kod noktasını ve UTF-8 byte'larını yazdır. Hangi harfler 1 byte, hangileri 2 byte tutuyor? Kod noktalarını doğrudan vocab olarak kullanmak neden kötü bir fikir, kendi cümlelerinle yaz. (0:14:56 - 0:23:50)

In [ ]:
cumle = "İstanbul'da güneşli bir öğleden sonra Işıl çay içti."

for ch in cumle:
    b = ch.encode('utf-8')
    print(f"{ch!r:6} kod noktası: {ord(ch):5} (U+{ord(ch):04X})  utf-8: {str(list(b)):12} -> {len(b)} byte")

print()
print('karakter sayısı:', len(cumle))
print('byte sayısı    :', len(cumle.encode('utf-8')))

In [ ]:
#### ascii'deki her sey 1 byte: ingilizce harfler, bosluk, noktalama.
# turkce harfler (ç ğ ı İ ö ş ü) 2 byte cunku ascii'nin ilk 128 karakterinin disindalar.
# ilginc olan: buyuk 'I' ascii'de var (1 byte) ama kucugu 'ı' yok (2 byte). kucuk 'i' 1 byte, buyugu 'İ' 2 byte.
# yani ayni harfin buyugu ve kucugu farkli uzunlukta olabiliyor. ayni cumlenin turkcesi ingilizcesinden daha cok byte tutuyor.
#
# kod noktalarini direkt vocab olarak kullanmak neden kotu:
# 1) unicode'da ~150 bin karakter var. embedding tablosu ve cikis katmani 150 bin satir olur,
#    cogu (eski alfabeler, semboller) egitimde neredeyse hic gorulmez, bosuna parametre ve kotu ogrenilmis satirlar.
# 2) unicode standardi guncelleniyor, yeni karakter eklendikce vocab da degismek zorunda kalir.
# 3) hala harf seviyesindeyiz: dizi uzun, block_size'a az metin sigiyor.
# utf-8 byte'lari ise sadece 256 farkli deger, kucuk ve sabit bir baslangic vocab'i. ama byte seviyesinde dizi daha da uzuyor
# (turkce harf 2 byte). bu yuzden BPE ile sik gecen byte gruplarini tek token yapiyoruz.

In [ ]:
#### neden harf seviyesi yetmedi de kelime parcasi (subword) kullaniliyor?
# iki farkli sayi var ve ters yonde hareket ediyorlar:
#   - vocab boyutu: kac FARKLI token var (sozluk)
#   - dizi uzunlugu: metin kac token'a bolunuyor
# ornek, kendi verim: nutuk 1.115.023 karakter, 102 farkli karakter.
#   harf seviyesinde -> vocab 102 (cok kucuk) ama dizi 1.115.023 adim (cok uzun).
#   kelime seviyesinde -> dizi kisa ama vocab yuz binlerce, ve "Ankara'dakilerden" gibi hic gorulmemis kelimede model caresiz.
#   kelime parcasi (BPE) -> ikisinin ortasi: sik gecen harf gruplari tek token, nadir kelime yine parcalardan kurulur.
#
# vocab boyutu modelin GIRISINI ve CIKISINI etkiliyor:
#   - embedding tablosu = vocab x n_embd. n_embd=384 ile: 102 x 384 ~ 39 bin parametre, 100.277 x 384 ~ 38,5 milyon parametre.
#   - cikis katmani her adimda vocab kadar secenek arasindan seciyor: 102 ucuz, 100 bin pahali.
#   - vocab buyudukce nadir token'lar egitimde az goruluyor ve kotu ogreniliyor.
#   - token buyudukce model token'in icindeki harfleri gormuyor (heceleme, harf sayma hatalari, task 5'in sonu).
# dizi uzunlugu modelin ICINI etkiliyor:
#   - block_size sabit. 256 adimlik pencereye harf seviyesinde ~30-40 kelime sigar, token basina 3-4 harf varsa 3-4 kat fazlasi.
#   - attention'da her token her token'a bakiyor: dizi 2 kat uzayinca is 4 kat artiyor.
#   - uretimde her token icin bir ileri gecis: uzun dizi = yavas uretim.
#   - harf seviyesinde model once harflerden kelime kurmayi ogrenmek zorunda, kapasitesinin bir kismi oraya gidiyor.
# sonuc: darbogaz vocab degil dizi uzunlugu. bu yuzden vocab'i tasinabilir kadar buyutup diziyi kisaltiyoruz.
#
# tiktokenizer'da ayni metin gpt-2 ile 300, gpt-4 ile 185 token cikiyordu: bu sayi vocab degil DIZI UZUNLUGU.
# ayni metin harf seviyesinde muhtemelen 1000'in uzerinde token olurdu (her harf bir token).

## task 2
BPE'yi sıfırdan yaz ve kendi Türkçe metninde eğit. Geçen hafta derlediğin Türkçe metni kullan. Ardışık ikilileri say, en sık geçeni yeni bir token'la birleştir, istediğin vocab boyutuna gelene kadar tekrarla. İlk 20 birleştirmeyi metin olarak yazdır: hangi harf grupları önce birleşiyor? Sıkıştırma oranını (byte sayısı / token sayısı) yaz. (0:23:50 - 0:39:20)

In [ ]:
# gecen haftaki nutuk metni (colab'de calisiyorsan input_tr.txt'yi yukle)
with open('input_tr.txt', 'r', encoding='utf-8') as f:
    text = f.read()

tokens = list(text.encode('utf-8'))
print('karakter sayısı:', len(text))
print('byte sayısı    :', len(tokens))

In [ ]:
def get_stats(ids):
    # ardisik ikilileri say: [1, 2, 3, 1, 2] -> {(1, 2): 2, (2, 3): 1, (3, 1): 1}
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx):
    # ids icindeki her 'pair' ikilisini tek token 'idx' ile degistir
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

print(get_stats([1, 2, 3, 1, 2]))
print(merge([5, 6, 6, 7, 9, 1], (6, 7), 99))  # [5, 6, 99, 9, 1]

In [ ]:
import time

vocab_size = 512          # 256 byte + 256 birlestirme
num_merges = vocab_size - 256

ids = list(tokens)        # orijinal byte listesini bozmamak icin kopya
merges = {}               # (int, int) -> int
vocab = {i: bytes([i]) for i in range(256)}
uzunluklar = [len(ids)]   # her birlestirmeden sonra dizi uzunlugu

t0 = time.time()
for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)   # en sik gecen ikili
    idx = 256 + i
    ids = merge(ids, pair, idx)
    merges[pair] = idx
    vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
    uzunluklar.append(len(ids))
print(f'egitim suresi: {time.time() - t0:.0f} sn')

In [ ]:
def goster(b):
    # byte'lari okunur yap: yarim kalan utf-8 karakterleri � olarak gorunur
    return b.decode('utf-8', errors='replace')

for (p0, p1), idx in list(merges.items())[:20]:
    print(f'{idx}: {goster(vocab[p0])!r:8} + {goster(vocab[p1])!r:8} -> {goster(vocab[idx])!r}')

print()
print('byte sayısı :', len(tokens))
print('token sayısı:', len(ids))
print(f'sıkıştırma oranı: {len(tokens) / len(ids):.2f}X')

In [ ]:
#### (calistirinca kendi gozlemini yaz) ilk birlesmelerde bakilacaklar:
# - turkce harflerin iki byte'i birlesip tek token oluyor mu? (ı = 196+177, ş = 197+159, ü = 195+188)
#   BPE once byte'lara bolunmus harfi geri kuruyor, sonra harf gruplarina geciyor.
# - bosluklu birlesmeler ('n ', 'e ', ' b' gibi): kelime sinirini asan token'lar var mi? regex olmadan bu oluyor, task 4'te bakacagiz.
# - turkce ekler ('ler', 'in', 'ar', 'dan') erken mi geliyor?
# sikistirma orani: ayni metin kac kat az adimla ifade ediliyor. block_size'a o kadar kat fazla metin sigar.

In [ ]:
# vocab buyudukce dizi nasil kisaliyor: her birlestirme vocab'a 1 token ekliyor, diziden bircok adim siliyor
print(f"{'birlestirme':>12}{'vocab':>8}{'token sayisi':>14}{'oran':>8}")
for k in [0, 1, 2, 4, 8, 16, 32, 64, 128, num_merges]:
    print(f'{k:>12}{256 + k:>8}{uzunluklar[k]:>14,}{len(tokens) / uzunluklar[k]:>7.2f}X')
print()
print('ilk birlestirme diziden', f'{uzunluklar[0] - uzunluklar[1]:,}', 'adim sildi, son birlestirme', f'{uzunluklar[-2] - uzunluklar[-1]:,}', 'adim sildi')

In [ ]:
#### sebep -> sonuc: vocab boyutu bir HIPERPARAMETRE, BPE onu uygulayan yontem.
# kac birlestirme yapacagima ben karar veriyorum (vocab_size = 512 -> 256 birlestirme).
# her birlestirme: vocab +1 token, dizi -(o ikilinin metinde gectigi sayi kadar) adim.
# ilk birlestirmeler en sik ikilileri yakaladigi icin cok adim siliyor, sonrakiler giderek daha az: azalan getiri.
# devam etseydim vocab buyur, dizi kisalir; erken dursam tersi.
#
# gpt-2 ve gpt-4 de ayni yontemi (byte seviyesinde BPE + once regex ile bolme) kullaniyor, sadece cok daha buyuk olcekte:
#   benim tokenizer'im : 512     = 256 byte + 256 birlestirme
#   gpt-2              : 50.257  = 256 byte + 50.000 birlestirme + 1 ozel token (<|endoftext|>)
#   gpt-4 (cl100k)     : 100.277 = ~100 bin birlestirme + birkac ozel token
# gpt-2 -> gpt-4'te yontem degismedi, uc malzeme degisti:
#   1) birlestirme sayisi ~2 katina cikti (nerede duracaklarina openai karar verdi)
#   2) tokenizer'in egitildigi metin: daha cok kod ve daha cok dil. birlestirmeler metne bagli:
#      ben nutuk'ta egittigim icin 'ı' ve osmanlica 'â' erken birlesiyor (30 bin karakterlik denemede ilk 5'teydiler);
#      gpt-4'te ise python girintisindeki bosluk gruplari tek token.
#      ayni 100 bin birlestirme baska bir metinde bambaska bir vocab cikarirdi.
#   3) regex deseni: kisaltmalari buyuk/kucuk harf ayirmadan yakaliyor, sayilari en fazla 3 haneli parcalara boluyor.
# tiktokenizer'daki 300 -> 185 dususu bunlarin toplam etkisi.
#
# neden 100 bin de daha az degil: model buyudukce embedding + cikis katmani toplam parametrenin kucuk bir kismi oluyor,
# buyuk model buyuk vocab'i rahat tasiyor ve karsiliginda daha kisa dizi kazaniyor.
# benim gecen haftaki kucuk gpt'm icin 100 bin satirlik embedding tablosu modelin kendisinden buyuk olurdu, 512 mantikli.

## task 3
encode ve decode yaz. decode(encode(metin)) metnin kendisini vermeli, Türkçe karakterli ve emoji içeren bir örnekle göster. Bazı token dizileri geçerli UTF-8 değil: decode bu durumda ne yapmalı, neden? (0:42:47 - 0:57:36)

In [ ]:
def encode(text, merges):
    ids = list(text.encode('utf-8'))
    while len(ids) >= 2:
        stats = get_stats(ids)
        # birlestirme tablosunda en once ogrenilmis (indeksi en kucuk) ikiliyi sec
        pair = min(stats, key=lambda p: merges.get(p, float('inf')))
        if pair not in merges:
            break   # birlestirilecek ikili kalmadi
        ids = merge(ids, pair, merges[pair])
    return ids

def decode(ids, vocab):
    b = b''.join(vocab[i] for i in ids)
    return b.decode('utf-8', errors='replace')

In [ ]:
ornek = "Çığ düştü, İstanbul'da şoför öğlen çay içti 🚀🙂"
ids_ornek = encode(ornek, merges)

print(ids_ornek)
print('karakter:', len(ornek), ' byte:', len(ornek.encode('utf-8')), ' token:', len(ids_ornek))
print(decode(ids_ornek, vocab))
print('decode(encode(x)) == x :', decode(ids_ornek, vocab) == ornek)

In [ ]:
# 128 = 0b10000000: utf-8'de "devam byte'i", tek basina bir karakter baslatamaz
print('replace:', decode([128], vocab))

try:
    vocab[128].decode('utf-8')   # varsayilan errors='strict'
except UnicodeDecodeError as e:
    print('strict :', e)

In [ ]:
#### emojiler utf-8'de 4 byte. nutuk'ta hic emoji yok, o yuzden birlesmiyorlar, 4 ayri byte token olarak kaliyorlar.
# yine de decode(encode(x)) == x cunku her sey byte'a iniyor, hicbir karakter kaybolmuyor (vocab disi karakter diye bir sey yok).
#
# gecersiz dizi: 128-191 arasi byte'lar devam byte'i, bir karakterin ortasinda olabilirler ama karakter baslatamazlar.
# model uretirken herhangi bir token dizisini cikarabilir, mesela bir emojinin sadece ilk 2 byte'ini.
# decode bu durumda patlamamali: errors='replace' gecersiz kismi � (U+FFFD) ile degistiriyor, metnin geri kalani saglam kaliyor.
# strict birakirsak uretimin ortasinda program UnicodeDecodeError ile cokuyor.

## task 4
GPT-4'ün regex desenini ekle. Metni önce GPT-4'ün deseniyle parçalara böl, BPE'yi her parçada ayrı çalıştır. Tokenizer'ı yeniden eğit, birleştirmeleri öncekiyle karşılaştır. Sonra "Ankara'dan" ve "İstanbul'da" gibi kesme işaretli kelimeleri bu desenle böl. Desen İngilizce için yazılmış: Türkçede ne yapıyor, kendi cümlelerinle yaz. (0:57:36 - 1:11:38)

In [ ]:
import regex as re   # \p{L} gibi unicode siniflari icin standart 're' yetmiyor

GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

print(re.findall(GPT4_SPLIT_PATTERN, "Hello've world123 how's are you!!!?"))

In [ ]:
from collections import Counter

chunks = re.findall(GPT4_SPLIT_PATTERN, text)
# ayni parca (' ve', ' bir' gibi) binlerce kez geciyor: her farkli parcayi bir kez tutup kac kez gectigiyle agirliklandiriyoruz.
# sonuc tek tek saymakla ayni, sadece cok daha hizli.
chunk_counts = Counter(chunks)
parca_ids = [list(ch.encode('utf-8')) for ch in chunk_counts]
parca_sayi = list(chunk_counts.values())
print('parca sayisi:', len(chunks), ' farkli parca:', len(parca_ids))

merges_re = {}
vocab_re = {i: bytes([i]) for i in range(256)}

t0 = time.time()
for i in range(num_merges):
    stats = {}
    for p_ids, n in zip(parca_ids, parca_sayi):
        for pair in zip(p_ids, p_ids[1:]):       # ikililer parcanin icinde kaliyor, parca sinirini asmiyor
            stats[pair] = stats.get(pair, 0) + n
    pair = max(stats, key=stats.get)
    idx = 256 + i
    parca_ids = [merge(p_ids, pair, idx) for p_ids in parca_ids]
    merges_re[pair] = idx
    vocab_re[idx] = vocab_re[pair[0]] + vocab_re[pair[1]]
print(f'egitim suresi: {time.time() - t0:.0f} sn')

In [ ]:
def encode_re(text, merges):
    # once desenle bol, her parcayi ayri encode et (ayni parcayi bir kez encode edip tekrar kullan)
    cache = {}
    ids = []
    for ch in re.findall(GPT4_SPLIT_PATTERN, text):
        if ch not in cache:
            cache[ch] = encode(ch, merges)
        ids.extend(cache[ch])
    return ids

ids_re = encode_re(text, merges_re)

print(f"{'':5}{'regexsiz':14}{'gpt-4 regex':14}")
for idx in range(256, 276):
    print(f'{idx}: {goster(vocab[idx])!r:14}{goster(vocab_re[idx])!r:14}')

sadece_basic = {vocab[i] for i in range(256, vocab_size)} - {vocab_re[i] for i in range(256, vocab_size)}
print()
print('sadece regexsiz tokenizer\'da olan token sayisi:', len(sadece_basic))
print('ornekler:', [goster(b) for b in sorted(sadece_basic, key=len, reverse=True)[:15]])
print()
print(f'sıkıştırma oranı (regexsiz)   : {len(tokens) / len(ids):.2f}X')
print(f'sıkıştırma oranı (gpt-4 regex): {len(tokens) / len(ids_re):.2f}X')

In [ ]:
#### (calistirinca kendi gozlemini yaz) regexsiz tokenizer'da 'n b', 'e k' gibi kelime sinirini asan token'lar olusabiliyor:
# iki kelimenin tesadufen yan yana sik gelmesi vocab'da yer kapliyor. regex ile bolunce birlesmeler kelimenin icinde kaliyor,
# bosluk kelimenin basina yapisiyor (' ve', ' bir'). sikistirma biraz dusebilir ama token'lar daha anlamli.

In [ ]:
for kelime in ["Ankara'dan", "İstanbul'da", "Türkiye'nin", "Atatürk'ün", "Ankara’dan", "1923'te", "I'm", "it's", "they'll"]:
    print(f'{kelime!r:16} -> {re.findall(GPT4_SPLIT_PATTERN, kelime)}')

print()
print("metindeki duz kesme  (')  :", text.count("'"))
print("metindeki kivrik kesme (’):", text.count('’'))

In [ ]:
#### desenin ilk parcasi '(?i:[sdmt]|ll|ve|re) ingilizce kisaltmalar icin: I'm, it's, they'll, we've.
# turkcede kesme isaretinden sonra ek geliyor ve ek bu harflerden biriyle baslarsa desen eki ortadan boluyor:
# "Ankara'dan" -> 'Ankara', "'d", 'an'   ve   "İstanbul'da" -> 'İstanbul', "'d", 'a'   ("'dan" eki ikiye ayriliyor)
# ama "Türkiye'nin" -> 'Türkiye', "'nin" ve "Atatürk'ün" -> 'Atatürk', "'ün" butun kaliyor.
# yani ayni tur ek (bulunma/ayrilma/tamlama eki) hangi harfle basladigina gore farkli bolunuyor: tutarsiz.
# "1923'te" -> '192', '3', "'t", 'e': sayilar 3'erli bolunuyor, ek de yine 't' yuzunden ikiye ayriliyor.
#
# nutuk'ta duz kesme (') hic yok, hepsi kivrik (’). desen kivrik kesmeyi kisaltma saymiyor: "Ankara’dan" -> 'Ankara', '’dan'.
# yani benim egitim verimde bu kural hic devreye girmiyor, ama kullanici duz ' ile yazarsa tokenizer ayni kelimeyi farkli bolecek.

## task 5
Türkçe ile İngilizceyi karşılaştır. Aynı paragrafın Türkçesini ve İngilizcesini al. GPT-4'ün tokenizer'ıyla (tiktoken, "cl100k_base") ikisini de token'la, kelime başına kaç token düştüğünü yaz. Sonra aynı Türkçe paragrafı kendi tokenizer'ınla token'la. Türkçenin eklemeli yapısı bu sayıları nasıl etkiliyor?

In [ ]:
import tiktoken
enc = tiktoken.get_encoding('cl100k_base')

tr = ("Yapay zeka modelleri metni doğrudan okumaz; önce onu küçük parçalara ayırır. "
      "Bu parçalara token denir. Türkçede bir kelimeye art arda birçok ek gelebildiği için "
      "aynı anlam İngilizceye göre daha az ama daha uzun kelimelerle anlatılır. "
      "Örneğin evlerimizden tek bir kelimedir, İngilizcede ise from our houses diye üç kelimeyle söylenir.")
en = ("Artificial intelligence models do not read text directly; they first split it into small pieces. "
      "These pieces are called tokens. In Turkish, many suffixes can be attached to a single word one after another, "
      "so the same meaning is expressed with fewer but longer words than in English. "
      "For example, evlerimizden is a single word, while in English it is said with three words: from our houses.")

def ozet(ad, metin, ids):
    kelime = len(metin.split())
    print(f'{ad:34} kelime: {kelime:3}  token: {len(ids):3}  token/kelime: {len(ids) / kelime:.2f}')

ozet('gpt-4 (cl100k) ingilizce', en, enc.encode(en))
ozet('gpt-4 (cl100k) turkce', tr, enc.encode(tr))
ozet('benim tokenizer (regexsiz) turkce', tr, encode(tr, merges))
ozet('benim tokenizer (regex) turkce', tr, encode_re(tr, merges_re))

In [ ]:
def gpt4_parcalar(s):
    return [enc.decode_single_token_bytes(t).decode('utf-8', errors='replace') for t in enc.encode(s)]

def benim_parcalar(s):
    return [goster(vocab_re[t]) for t in encode_re(s, merges_re)]

for s in [' evlerimizden', ' from our houses', 'İstanbul', 'strawberry', ' strawberry']:
    print(f'{s!r:18} gpt-4: {gpt4_parcalar(s)}')
print()
for s in [' evlerimizden', ' İstanbul', ' Türkiye']:
    print(f'{s!r:18} benim: {benim_parcalar(s)}')

In [ ]:
#### gpt-4'te ingilizce kelime basina 1.24 token, turkce 2.57 token: ayni anlam icin turkce ~2 kat token harciyor.
# sebep 1: eklemeli yapi. 'evlerimizden' tek kelime ama gpt-4 onu 'ev' 'ler' 'im' 'iz' 'den' diye 5 parcaya boluyor,
# ingilizcesi 'from our houses' 3 kelime, 3 token. turkcede kelime sayisi az ama her kelime cok parcaya bolunuyor.
# sebep 2: gpt-4'un tokenizer'i agirlikla ingilizce metinle egitilmis. ' houses' tek token, turkce kelimelerin cogu tek token degil.
# turkce harfler 2 byte oldugu icin de daha az birlesiyor ('İstanbul' -> 'İ' + 'stanbul').
# sonuc: turkce konusan kullanici ayni context penceresine daha az metin sigdiriyor ve api'de ayni anlam icin daha cok oduyor.
#
# (calistirinca yaz) benim tokenizer'im sadece 256 birlestirme ile egitildi ve nutuk'un eski turkcesinden ogrendi,
# o yuzden modern turkce paragrafta kelime basina token sayisi ... cikti.
# vocab'i buyutsem (ör. 2000) ve modern turkce metinle egitsem sayi duserdi.
#
# strawberry: bosluksuz 'str' 'aw' 'berry', basinda bosluk varken tek token. model kelimenin icindeki harfleri gormuyor,
# llm'lerin 'strawberry'de kac r var' sorusunda yanilmasinin sebebi bu.

In [ ]:
# model harf sayamiyorsa nasil dogru cevap verebiliyor? kelimeyi harf harf acinca her harf ayri token oluyor
for s in ['strawberry', 's-t-r-a-w-b-e-r-r-y', 's t r a w b e r r y']:
    print(f'{s!r:24} {len(gpt4_parcalar(s)):2} token: {gpt4_parcalar(s)}')
print()
print(re.findall(GPT4_SPLIT_PATTERN, 'a-l-i-n'))   # tire harfe yapissa bile her parcada tek harf kaliyor

In [ ]:
#### sebep -> sonuc: tokenizer harf saymayi neden zorlastiriyor, model bunu nasil asiyor
# 'berry' tek token ise model ona tek bir sayi olarak bakiyor, icinde iki 'r' oldugunu dogrudan goremiyor.
# bunu ancak egitim verisinde "berry iki r ile yazilir" gibi bir seyle karsilastiysa ogrenebiliyor (ezber).
# ikinci yol: model kelimeyi s-t-r-a-w-b-e-r-r-y diye acip yaziyor. araya giren tire/bosluk BPE'nin harfleri
# birlestirmesini engelliyor (BPE sadece yan yana duran parcalari birlestirir), her harf ayri token oluyor ve sayma gorunur hale geliyor.
# bunu tokenizer yapmiyor: tokenizer sabit bir tablo. harf harf acmaya model karar veriyor, egitimde ogrenilmis bir davranis.
# hata tamamen yok olmadi: uzun, nadir ya da uydurma kelimelerde harf harf acmadan cevap verilirse hala yanilabiliyor.
#
# tokenizer modeli akilli yapan sey mi? cogunlukla hayir. tokenizer modelin metni okudugu alfabe gibi:
# verimliligi (pencereye ne kadar metin sigar, maliyet) ve bazi tuhafliklari (heceleme, sayilar, az temsil edilen diller) belirliyor.
# modeller arasindaki asil farki olcek (parametre + hesap), veri kalitesi ve on egitim sonrasi egitim yaratiyor.
# gpt-2 -> gpt-4'te tokenizer da degisti ve kodda verimlilik artti, ama buyuk sicramayi tokenizer degil olcek ve egitim yapti.